# Lesson 10.2 - Perception, Planning & Navigation (gridworld demo)

## Objectives

- Implement grid-based planning with A*.
- Visualize planned path around obstacles.
- Connect discrete planning to real robot navigation stacks.


In [ ]:
from __future__ import annotations

import heapq
import numpy as np
import matplotlib.pyplot as plt


## Gridworld Definition

- `0`: free cell
- `1`: obstacle

Start and goal are placed on free cells.


In [ ]:
grid = np.array([
    [0,0,0,0,0,0,0,0,0,0],
    [0,1,1,1,0,0,1,1,1,0],
    [0,0,0,1,0,0,0,0,1,0],
    [1,1,0,1,0,1,1,0,1,0],
    [0,0,0,0,0,1,0,0,0,0],
    [0,1,1,1,0,1,0,1,1,0],
    [0,0,0,1,0,0,0,1,0,0],
    [0,1,0,1,1,1,0,1,0,1],
    [0,1,0,0,0,0,0,0,0,0],
    [0,0,0,1,1,0,1,1,1,0],
], dtype=int)

start = (0, 0)
goal = (9, 9)


## A* Implementation


In [ ]:
def heuristic(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])  # Manhattan distance


def astar(grid, start, goal):
    rows, cols = grid.shape
    neighbors = [(1,0), (-1,0), (0,1), (0,-1)]

    open_heap = [(0 + heuristic(start, goal), 0, start)]
    came_from = {start: None}
    g_score = {start: 0}

    while open_heap:
        _, g, current = heapq.heappop(open_heap)
        if current == goal:
            # reconstruct path
            path = []
            c = current
            while c is not None:
                path.append(c)
                c = came_from[c]
            return path[::-1]

        for dr, dc in neighbors:
            nr, nc = current[0] + dr, current[1] + dc
            if nr < 0 or nr >= rows or nc < 0 or nc >= cols:
                continue
            if grid[nr, nc] == 1:
                continue

            nxt = (nr, nc)
            tentative_g = g + 1
            if nxt not in g_score or tentative_g < g_score[nxt]:
                g_score[nxt] = tentative_g
                came_from[nxt] = current
                f = tentative_g + heuristic(nxt, goal)
                heapq.heappush(open_heap, (f, tentative_g, nxt))

    return None


path = astar(grid, start, goal)
print('Path length:', len(path) if path else None)


In [ ]:
def visualize(grid, path, start, goal):
    img = np.where(grid == 1, 0.0, 1.0)
    plt.figure(figsize=(6, 6))
    plt.imshow(img, cmap='gray', origin='upper')

    if path is not None:
        py = [p[0] for p in path]
        px = [p[1] for p in path]
        plt.plot(px, py, color='royalblue', linewidth=2, label='Path')

    plt.scatter([start[1]], [start[0]], color='green', s=80, label='Start')
    plt.scatter([goal[1]], [goal[0]], color='red', s=80, label='Goal')

    plt.title('Gridworld A* Path')
    plt.legend(loc='upper right')
    plt.xticks(range(grid.shape[1]))
    plt.yticks(range(grid.shape[0]))
    plt.grid(True, alpha=0.2)
    plt.show()


visualize(grid, path, start, goal)


## Connect to Theory

- This grid is a simplified occupancy map.
- A* is a global planner on discrete state space.
- Real robots pair global planners with local controllers and live sensor updates.


## Business Case Studies & Exceptions

### Case A: Autonomous Delivery Carts

Global routes can be planned with map-based search, but local behavior must react to people, pets, and temporary obstacles.

### Case B: Factory AMRs

A* works for fixed maps, while fleet orchestration handles traffic conflicts across many robots.

### Exceptions

- In highly dynamic scenes, purely deliberative path planning may be too slow without reactive layers.
- In continuous environments, graph planners need smoothing and control-aware trajectory generation.


## Interview Questions & Answers

1. **What problem does A* solve?**  
   It finds a low-cost path from start to goal on a graph.
2. **What is the role of heuristic in A*?**  
   It estimates remaining cost to guide search efficiently.
3. **Why is Manhattan distance used here?**  
   It is admissible for 4-neighbor grid movement.
4. **What is occupancy mapping?**  
   Representing environment cells as free/occupied/unknown.
5. **How is SLAM related to planning?**  
   SLAM provides map/pose needed for planning.
6. **Global vs local planning?**  
   Global sets route; local handles short-term obstacle avoidance.
7. **What is a recovery behavior?**  
   A fallback strategy when planner/controller gets stuck.
8. **How do sensor errors affect navigation?**  
   They can create false obstacles or bad pose estimates, causing path failures.
9. **Why are safety buffers used?**  
   To maintain clearance around obstacles under uncertainty.
10. **What changes when moving from gridworld to real robots?**  
   Continuous dynamics, latency, sensor fusion, and actuation constraints.
